# Phase 1 — Data Ingestion and Analysis Spec

**Project:** GradeLens — Habit-Based Academic Performance Analysis  
**Notebook:** `01_data_ingestion_and_spec.ipynb`  
**Purpose:** Load the raw dataset, build the data dictionary, define the analysis spec (outcome, performance bands, column roles), and write the processed schema contract for the team.

---

### Dataset source

**Name:** College Students Habits & Performance (1M rows)  
**File:** `data/archive/college_students_habits_1M.csv`  
**Source:** https://www.kaggle.com/datasets/

> Raw file is kept unmodified. All processing happens in later phases.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW_PATH = "../data/archive/college_students_habits_1M.csv"
RAW_OUT   = "../data/raw/college_students_habits_1M.csv"
PROC_DIR  = "../data/processed/"
FIG_DIR   = "../figures/"

import shutil, os
os.makedirs("../data/raw",       exist_ok=True)
os.makedirs(PROC_DIR,            exist_ok=True)
os.makedirs(FIG_DIR,             exist_ok=True)

# Copy raw file into data/raw/ as the immutable snapshot
if not os.path.exists(RAW_OUT):
    shutil.copy2(RAW_PATH, RAW_OUT)
    print("Raw snapshot saved to", RAW_OUT)
else:
    print("Raw snapshot already exists:", RAW_OUT)

Raw snapshot already exists: ../data/raw/college_students_habits_1M.csv


## 1. Load and initial inspection

In [3]:
df = pd.read_csv(RAW_OUT)

print("Shape:", df.shape)
print(f"  {df.shape[0]:,} rows  |  {df.shape[1]} columns")

Shape: (1000000, 42)
  1,000,000 rows  |  42 columns


In [4]:
df.dtypes.to_frame(name="dtype")

,dtype
study_hours,float64
attendance,float64
assignment_completion,float64
midterm_score,float64
final_score,float64
project_score,float64
backlogs,int64
sleep_hours,float64
stress,float64
anxiety,float64


In [5]:
df.head(5)

,study_hours,attendance,assignment_completion,midterm_score,final_score,project_score,backlogs,sleep_hours,stress,anxiety,...,hostel_student,extracurricular_hours,phone_unlocks_per_day,previous_gpa,class_participation,weekly_study_sessions,group_study_hours,financial_stress,gpa,performance_level
0,3.014684,67.00599,51.595387,57.211285,61.653540,65.397200,4,5.993893,4.287966,58.146000,...,0,1.958940,73.727480,5.721128,3.587111,2.814086,1.814086,5.491878,0.546729,Low
1,3.665277,73.28455,69.749020,57.552320,62.062782,65.715500,3,6.949383,1.841224,41.945290,...,0,3.146447,48.468456,5.755232,4.820090,2.836821,1.836821,2.876881,0.707133,Low
2,2.703784,72.32519,92.837640,44.568970,46.482760,53.597702,2,6.703293,3.863112,56.555750,...,1,5.551245,46.623684,4.456897,5.493774,1.971265,0.971265,5.704047,0.868230,Low
3,3.445073,74.75687,85.189026,52.040790,55.448948,60.571404,2,6.498832,5.073206,65.171420,...,1,4.543216,47.909600,5.204079,5.481987,2.469386,1.469386,6.596658,0.729216,Low
4,0.192687,55.05021,64.520620,32.815000,32.378000,42.627330,5,6.552570,1.000000,30.725826,...,1,4.447042,73.316520,3.281500,2.822375,1.187667,0.187667,4.602954,0.370964,Low


In [6]:
# Missing values overview
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct}).query("missing_count > 0")

,missing_count,missing_pct
performance_level,1558,0.16


In [7]:
# Unique value counts per column — helps distinguish continuous, discrete, and binary columns
pd.DataFrame({
    "dtype":   df.dtypes,
    "n_unique": df.nunique(),
    "sample_values": {c: df[c].dropna().unique()[:5].tolist() for c in df.columns}
})

,dtype,n_unique,sample_values
study_hours,float64,938383,"[3.0146842, 3.6652765, 2.7037838, 3.4450727, 0..."
attendance,float64,898793,"[67.00599, 73.28455, 72.32519, 74.75687, 55.05..."
assignment_completion,float64,919009,"[51.595387, 69.74902, 92.83764, 85.189026, 64...."
midterm_score,float64,949846,"[57.211285, 57.55232, 44.56897, 52.04079, 32.815]"
final_score,float64,930318,"[61.65354, 62.062782, 46.48276, 55.448948, 32...."
project_score,float64,929057,"[65.3972, 65.7155, 53.597702, 60.571404, 42.62..."
backlogs,int64,11,"[4, 3, 2, 5, 1]"
sleep_hours,float64,862188,"[5.993893, 6.949383, 6.703293, 6.4988317, 6.55..."
stress,float64,826386,"[4.287966, 1.841224, 3.8631122, 5.0732064, 1.0]"
anxiety,float64,961981,"[58.146, 41.94529, 56.55575, 65.17142, 30.725826]"


In [8]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
study_hours,1000000.0,4.042036,2.212739,0.000000,2.457050,4.004593,5.552457,12.000000
attendance,1000000.0,74.876365,12.892864,30.000000,66.092450,75.027325,83.971757,100.000000
assignment_completion,1000000.0,69.896701,14.682793,20.000000,59.909055,70.021207,80.136895,100.000000
midterm_score,1000000.0,60.011309,14.967273,0.000000,49.891078,60.027573,70.158226,100.000000
final_score,1000000.0,64.856105,17.606373,0.000000,52.869294,65.033085,77.189865,100.000000
project_score,1000000.0,67.972723,13.877038,0.000000,58.565006,68.025735,77.481010,100.000000
backlogs,1000000.0,2.550791,1.650993,0.000000,1.000000,2.000000,4.000000,10.000000
sleep_hours,1000000.0,6.499550,0.435650,4.520054,6.206206,6.498872,6.793280,8.763114
stress,1000000.0,3.211822,1.756919,1.000000,1.686328,3.046761,4.410195,10.000000
anxiety,1000000.0,49.986145,14.999785,0.000000,39.862051,49.973913,60.099820,100.000000


## 2. Data dictionary

| # | Column | Dtype | Description | Value range / categories | Notes |
|---|--------|-------|-------------|--------------------------|-------|
| 1 | `study_hours` | float | Daily hours spent studying | ~0–12 h | Core habit |
| 2 | `attendance` | float | Class attendance rate (%) | 0–100 | Core academic |
| 3 | `assignment_completion` | float | % of assignments submitted | 0–100 | Core academic |
| 4 | `midterm_score` | float | Midterm exam score (%) | 0–100 | Intermediate outcome |
| 5 | `final_score` | float | Final exam score (%) | 0–100 | Intermediate outcome |
| 6 | `project_score` | float | Project/coursework score (%) | 0–100 | Intermediate outcome |
| 7 | `backlogs` | int | Number of failed / pending subjects | 0+ | Academic risk indicator |
| 8 | `sleep_hours` | float | Nightly sleep duration (h) | ~3–10 h | Core habit |
| 9 | `stress` | float | Self-reported stress level | ~1–10 scale | Wellbeing |
| 10 | `anxiety` | float | Anxiety score | ~0–100 scale | Wellbeing |
| 11 | `depression` | float | Depression score | ~0–100 scale | Wellbeing |
| 12 | `motivation` | float | Self-reported motivation level | ~1–10 scale | Wellbeing |
| 13 | `concentration` | float | Self-reported concentration | ~1–10 scale | Wellbeing |
| 14 | `time_management` | float | Time management skill score | ~1–10 scale | ⚠️ Identical to `self_discipline` in sample — check correlation |
| 15 | `self_discipline` | float | Self-discipline score | ~1–10 scale | ⚠️ Identical to `time_management` in sample — check correlation |
| 16 | `social_media_hours` | float | Daily social media usage (h) | ~0–8 h | Screen habit |
| 17 | `gaming_hours` | float | Daily gaming hours | ~0–5 h | ⚠️ Identical to `netflix_hours` in sample — check correlation |
| 18 | `netflix_hours` | float | Daily streaming hours | ~0–5 h | ⚠️ Identical to `gaming_hours` in sample — check correlation |
| 19 | `screen_time` | float | Total daily screen time (h) | ~0–15 h | Likely sum of cols 16–18 — check for perfect linear dependency |
| 20 | `physical_activity` | float | Weekly physical activity (h) | ~0–10 h | Lifestyle |
| 21 | `junk_food_frequency` | float | Junk food meals per week | ~0–7 | Lifestyle |
| 22 | `caffeine_mg` | float | Daily caffeine intake (mg) | ~0–400 mg | Lifestyle |
| 23 | `late_night_frequency` | float | Late-night study/activity sessions per week | ~0–7 | ⚠️ Identical to `gaming_hours` / `netflix_hours` in sample — check |
| 24 | `procrastination_score` | float | Procrastination tendency | ~1–10 scale | Behaviour |
| 25 | `family_income` | float | Annual family income (currency units) | ~5,000–100,000 | Socioeconomic |
| 26 | `parental_education_level` | int | Ordinal: 1=Primary → 5=Postgrad | 1–5 | Socioeconomic |
| 27 | `internet_quality` | float | Internet quality/speed score | ~1–10 scale | Resource |
| 28 | `library_visits` | float | Library visits per week | ~0–5 | Academic resource |
| 29 | `online_courses_completed` | int | Number of extra online courses done | 0+ | Proactive learning |
| 30 | `part_time_hours` | float | Weekly part-time work hours | ~0–20 h | Time constraint |
| 31 | `peer_study_group` | int | Participates in peer study group (0/1) | 0, 1 | Binary |
| 32 | `relationship_status` | int | In a relationship (0/1) | 0, 1 | Binary |
| 33 | `hostel_student` | int | Lives in hostel/dorm (0/1) | 0, 1 | Binary |
| 34 | `extracurricular_hours` | float | Weekly extracurricular hours | ~0–10 h | Time allocation |
| 35 | `phone_unlocks_per_day` | float | Phone unlocks per day | ~10–120 | Distraction proxy |
| 36 | `previous_gpa` | float | Previous semester GPA | ~0–10 scale | ⚠️ Scale differs from `gpa` (~0–2). High leakage risk for habit association rules |
| 37 | `class_participation` | float | Class participation score | ~1–10 scale | Academic engagement |
| 38 | `weekly_study_sessions` | float | Number of study sessions per week | ~1–7 | Core habit |
| 39 | `group_study_hours` | float | Weekly group study hours | ~0–5 h | Collaborative learning |
| 40 | `financial_stress` | float | Financial stress score | ~1–10 scale | Stressor |
| 41 | `gpa` | float | **Target — current GPA** | 0–1 scale (normalised?) | Primary outcome |
| 42 | `performance_level` | object | Categorical performance label | Low / Medium / High | ⚠️ Drop — only contains one unique value ('Low') |


### 2a. Flagged columns — investigate before encoding

| Flag | Columns | Concern |
|------|---------|--------|
| Possible duplicates | `time_management`, `self_discipline` | Identical values in 3-row sample — likely near-perfect correlation; one may be dropped |
| Possible duplicates | `gaming_hours`, `netflix_hours`, `late_night_frequency` | Identical values in sample — check if they are truly different measurements |
| Derived / composite | `screen_time` | Likely = `social_media_hours + gaming_hours + netflix_hours`; if so, drop to avoid multicollinearity |
| Scale mismatch | `gpa` vs `previous_gpa` | `gpa` appears 0–1; `previous_gpa` appears 0–10; must normalise to same scale |
| Leakage risk | `midterm_score`, `final_score`, `project_score` | These are outcomes/components of GPA, not habits; handle carefully in association rules to avoid circular rules |


In [9]:
# Verify flagged duplicates — correlation between suspected identical columns
flag_pairs = [
    ("time_management", "self_discipline"),
    ("gaming_hours", "netflix_hours"),
    ("gaming_hours", "late_night_frequency"),
]
for a, b in flag_pairs:
    r = df[a].corr(df[b])
    print(f"corr({a}, {b}) = {r:.6f}")

corr(time_management, self_discipline) = 1.000000
corr(gaming_hours, netflix_hours) = 1.000000
corr(gaming_hours, late_night_frequency) = 1.000000


In [10]:
# Verify screen_time as a composite
computed = df["social_media_hours"] + df["gaming_hours"] + df["netflix_hours"]
diff = (df["screen_time"] - computed).abs()
print("Max difference between screen_time and sum of components:", diff.max())
print("Mean difference:", diff.mean())
# If max diff ~ 0, screen_time is fully derived and can be dropped

Max difference between screen_time and sum of components: 2.3999999996249244e-06
Mean difference: 2.2187892343710686e-07


In [11]:
# Verify GPA scale mismatch
print("gpa          — min:", df["gpa"].min().round(4), " max:", df["gpa"].max().round(4))
print("previous_gpa — min:", df["previous_gpa"].min().round(4), " max:", df["previous_gpa"].max().round(4))
print("\nperformance_level value counts:")
print(df["performance_level"].value_counts())

gpa          — min: 0.0  max: 2.0091
previous_gpa — min: 0.0  max: 10.0

performance_level value counts:
performance_level
Low    998442
Name: count, dtype: int64


## 3. Analysis spec

This section defines the analytical targets and column roles for the entire project. Phase 2 (preprocessing) and Person A (K-Means) should follow this spec.

---

### 3a. Primary outcome

| Item | Decision |
|------|----------|
| **Primary target** | `gpa` — continuous; used for regression, correlation, and EDA |
| **Dropped target** | `performance_level` — discarded due to having only one unique value ('Low') |
| **GPA scale note** | Confirm scale in cell above. If `gpa` is 0–1 and `previous_gpa` is 0–10, normalise `previous_gpa` to 0–1 in Phase 2 |

---

### 3b. Performance bands for association rules

Continuous `gpa` is discretised into 4 bands using **quartiles** (computed from the data below). Labels are kept interpretable.

| Band label | GPA quartile | Meaning |
|------------|-------------|--------|
| `gpa_low` | Q0 – Q1 (bottom 25%) | At-risk students |
| `gpa_medium_low` | Q1 – Q2 | Below-average performers |
| `gpa_medium_high` | Q2 – Q3 | Above-average performers |
| `gpa_high` | Q3 – Q4 (top 25%) | High performers |

Cut points are computed dynamically below to avoid hardcoding.

In [12]:
# Compute and print GPA quartile cut points
q1, q2, q3 = df["gpa"].quantile([0.25, 0.50, 0.75]).values
gpa_min = df["gpa"].min()
gpa_max = df["gpa"].max()

print("GPA distribution:")
print(f"  min           = {gpa_min:.4f}")
print(f"  Q1  (25%)     = {q1:.4f}   -> cut between gpa_low and gpa_medium_low")
print(f"  Q2  (50%)     = {q2:.4f}   -> cut between gpa_medium_low and gpa_medium_high")
print(f"  Q3  (75%)     = {q3:.4f}   -> cut between gpa_medium_high and gpa_high")
print(f"  max           = {gpa_max:.4f}")

GPA_BINS   = [gpa_min - 0.001, q1, q2, q3, gpa_max + 0.001]
GPA_LABELS = ["gpa_low", "gpa_medium_low", "gpa_medium_high", "gpa_high"]

print("\nBin edges (for pd.cut in Phase 4):", [round(b, 4) for b in GPA_BINS])
print("Bin labels:", GPA_LABELS)

GPA distribution:
  min           = 0.0000
  Q1  (25%)     = 0.6266   -> cut between gpa_low and gpa_medium_low
  Q2  (50%)     = 0.8305   -> cut between gpa_medium_low and gpa_medium_high
  Q3  (75%)     = 1.0342   -> cut between gpa_medium_high and gpa_high
  max           = 2.0091

Bin edges (for pd.cut in Phase 4): [np.float64(-0.001), np.float64(0.6266), np.float64(0.8305), np.float64(1.0342), np.float64(2.0101)]
Bin labels: ['gpa_low', 'gpa_medium_low', 'gpa_medium_high', 'gpa_high']


### 3c. Column roles

Every column is assigned one of four roles:

| Role | Meaning |
|------|---------|
| `target` | Primary or secondary outcome — not used as a feature |
| `numeric_feature` | Continuous or integer feature used in EDA, rules, and clustering |
| `categorical_feature` | Binary or ordinal feature that requires encoding |
| `drop` | Derived, redundant, or leakage-risk column — excluded after verification |

In [13]:
# Column roles — update after verifying flagged columns in section 2a
column_roles = {
    # Targets
    "gpa":                        "target",

    # Core academic behaviour
    "study_hours":                 "numeric_feature",
    "attendance":                  "numeric_feature",
    "assignment_completion":       "numeric_feature",
    "backlogs":                    "numeric_feature",
    "weekly_study_sessions":       "numeric_feature",
    "class_participation":         "numeric_feature",
    "library_visits":              "numeric_feature",
    "online_courses_completed":    "numeric_feature",
    "group_study_hours":           "numeric_feature",
    "previous_gpa":                "numeric_feature",  # normalise in Phase 2; DROP in Phase 4 (leakage risk)

    # Intermediate outcomes (leakage risk — keep but handle with care in rules)
    "midterm_score":               "numeric_feature",
    "final_score":                 "numeric_feature",
    "project_score":               "numeric_feature",

    # Lifestyle / health habits
    "sleep_hours":                 "numeric_feature",
    "physical_activity":           "numeric_feature",
    "junk_food_frequency":         "numeric_feature",
    "caffeine_mg":                 "numeric_feature",
    "late_night_frequency":        "drop",             # corr=1.0 with gaming_hours

    # Wellbeing / psychological
    "stress":                      "numeric_feature",
    "anxiety":                     "numeric_feature",
    "depression":                  "numeric_feature",
    "motivation":                  "numeric_feature",
    "concentration":               "numeric_feature",
    "procrastination_score":       "numeric_feature",
    "financial_stress":            "numeric_feature",
    "time_management":             "numeric_feature",
    "self_discipline":             "drop",             # corr=1.0 with time_management

    # Screen / distraction
    "social_media_hours":          "numeric_feature",
    "gaming_hours":                "numeric_feature",
    "netflix_hours":               "drop",             # corr=1.0 with gaming_hours
    "screen_time":                 "drop",              # likely derived — confirm with cell above
    "phone_unlocks_per_day":       "numeric_feature",

    # Time allocation
    "part_time_hours":             "numeric_feature",
    "extracurricular_hours":       "numeric_feature",

    # Socioeconomic
    "family_income":               "numeric_feature",
    "parental_education_level":    "categorical_feature",  # ordinal 1–5
    "internet_quality":            "numeric_feature",

    # Binary categorical
    "peer_study_group":            "categorical_feature",
    "relationship_status":         "categorical_feature",
    "hostel_student":              "categorical_feature",
}

roles_df = pd.DataFrame.from_dict(column_roles, orient="index", columns=["role"])
print(roles_df["role"].value_counts().to_string())
print(roles_df)


role
numeric_feature        32
drop                    4
categorical_feature     4
target                  1
                                         role
gpa                                    target
study_hours                   numeric_feature
attendance                    numeric_feature
assignment_completion         numeric_feature
backlogs                      numeric_feature
weekly_study_sessions         numeric_feature
class_participation           numeric_feature
library_visits                numeric_feature
online_courses_completed      numeric_feature
group_study_hours             numeric_feature
previous_gpa                  numeric_feature
midterm_score                 numeric_feature
final_score                   numeric_feature
project_score                 numeric_feature
sleep_hours                   numeric_feature
physical_activity             numeric_feature
junk_food_frequency           numeric_feature
caffeine_mg                   numeric_feature
late_night_freque

In [14]:
# Clustering inputs for Person A — all numeric_feature columns (unscaled names)
clustering_inputs = roles_df[roles_df["role"] == "numeric_feature"].index.tolist()
print(f"Clustering inputs ({len(clustering_inputs)} columns):")
for c in clustering_inputs:
    print(" -", c)

Clustering inputs (32 columns):
 - study_hours
 - attendance
 - assignment_completion
 - backlogs
 - weekly_study_sessions
 - class_participation
 - library_visits
 - online_courses_completed
 - group_study_hours
 - previous_gpa
 - midterm_score
 - final_score
 - project_score
 - sleep_hours
 - physical_activity
 - junk_food_frequency
 - caffeine_mg
 - stress
 - anxiety
 - depression
 - motivation
 - concentration
 - procrastination_score
 - financial_stress
 - time_management
 - social_media_hours
 - gaming_hours
 - phone_unlocks_per_day
 - part_time_hours
 - extracurricular_hours
 - family_income
 - internet_quality


## 4. Processed schema contract

> **Team contract** — Phase 2 preprocessing and Person A's K-Means implementation must follow this schema. Do not change column names after this point without updating this table.

| Column | Final dtype | Encoding plan | Notes |
|--------|-------------|---------------|-------|
| `study_hours` | float32 | passthrough | |
| `attendance` | float32 | passthrough | |
| `assignment_completion` | float32 | passthrough | |
| `midterm_score` | float32 | passthrough | Leakage risk in rules — exclude from habit-only rule sets |
| `final_score` | float32 | passthrough | Leakage risk in rules — exclude from habit-only rule sets |
| `project_score` | float32 | passthrough | Leakage risk in rules — exclude from habit-only rule sets |
| `backlogs` | int16 | passthrough | |
| `sleep_hours` | float32 | passthrough | |
| `stress` | float32 | passthrough | |
| `anxiety` | float32 | passthrough | |
| `depression` | float32 | passthrough | |
| `motivation` | float32 | passthrough | |
| `concentration` | float32 | passthrough | |
| `time_management` | float32 | passthrough | |
| `self_discipline` | — | **drop** | Identical to `time_management` (corr=1.0) |
| `social_media_hours` | float32 | passthrough | |
| `gaming_hours` | float32 | passthrough | |
| `netflix_hours` | — | **drop** | Identical to `gaming_hours` (corr=1.0) |
| `screen_time` | — | **drop** | Derived column — confirmed in section 2a |
| `physical_activity` | float32 | passthrough | |
| `junk_food_frequency` | float32 | passthrough | |
| `caffeine_mg` | float32 | passthrough | |
| `late_night_frequency` | — | **drop** | Identical to `gaming_hours` (corr=1.0) |
| `procrastination_score` | float32 | passthrough | |
| `family_income` | float32 | passthrough | |
| `parental_education_level` | int8 | **ordinal** (1–5, already encoded) | |
| `internet_quality` | float32 | passthrough | |
| `library_visits` | float32 | passthrough | |
| `online_courses_completed` | int16 | passthrough | |
| `part_time_hours` | float32 | passthrough | |
| `peer_study_group` | int8 | passthrough (already 0/1) | |
| `relationship_status` | int8 | passthrough (already 0/1) | |
| `hostel_student` | int8 | passthrough (already 0/1) | |
| `extracurricular_hours` | float32 | passthrough | |
| `phone_unlocks_per_day` | float32 | passthrough | |
| `previous_gpa` | float32 | **normalise** to 0–1 scale to match `gpa` | **Drop in Phase 4** to prevent rule leakage |
| `class_participation` | float32 | passthrough | |
| `weekly_study_sessions` | float32 | passthrough | |
| `group_study_hours` | float32 | passthrough | |
| `financial_stress` | float32 | passthrough | |
| `gpa` | float32 | passthrough | **Primary target** — do not scale |
| `performance_level` | — | **drop** | **Dropped target** — discarded due to having only one unique value ('Low') |

**Scaled version:** A separate file `students_scaled.csv` will contain `_scaled` suffixed versions of all `numeric_feature` columns (StandardScaler), for use by Person A's K-Means. The unscaled file (`students_clean.csv`) is used for all EDA and association rules.

In [15]:
# Summary printout — quick reference for Person A
print("=" * 55)
print("TEAM CONTRACT SUMMARY")
print("=" * 55)
print(f"  Raw file     : data/raw/college_students_habits_1M.csv")
print(f"  Clean output : data/processed/students_clean.csv      (Phase 2)")
print(f"  Scaled output: data/processed/students_scaled.csv     (Phase 2)")
print()
print(f"  Primary target   : gpa (float, 0-1 scale)")
print(f"  Dropped target   : performance_level (Only contains 'Low')")
print()
print(f"  Columns to DROP  : screen_time, performance_level, self_discipline, netflix_hours, late_night_frequency")


print(f"  Columns to NORMALISE: previous_gpa (to 0-1 scale)")
print()
print(f"  Clustering inputs ({len(clustering_inputs)} columns) — for Person A:")
for i, c in enumerate(clustering_inputs, 1):
    print(f"    {i:2d}. {c}")
print("=" * 55)

TEAM CONTRACT SUMMARY
  Raw file     : data/raw/college_students_habits_1M.csv
  Clean output : data/processed/students_clean.csv      (Phase 2)
  Scaled output: data/processed/students_scaled.csv     (Phase 2)

  Primary target   : gpa (float, 0-1 scale)
  Dropped target   : performance_level (Only contains 'Low')

  Columns to DROP  : screen_time, performance_level, self_discipline, netflix_hours, late_night_frequency
  Columns to NORMALISE: previous_gpa (to 0-1 scale)

  Clustering inputs (32 columns) — for Person A:
     1. study_hours
     2. attendance
     3. assignment_completion
     4. backlogs
     5. weekly_study_sessions
     6. class_participation
     7. library_visits
     8. online_courses_completed
     9. group_study_hours
    10. previous_gpa
    11. midterm_score
    12. final_score
    13. project_score
    14. sleep_hours
    15. physical_activity
    16. junk_food_frequency
    17. caffeine_mg
    18. stress
    19. anxiety
    20. depression
    21. motivation


## 5. Deliverable check

- [ ] `data/raw/college_students_habits_1M.csv` exists (immutable snapshot)
- [ ] `data/processed/` and `figures/` directories exist
- [ ] Data dictionary covers all 42 columns
- [ ] Flagged columns investigated with correlation checks (section 2a)
- [ ] GPA scale confirmed (section 3a)
- [ ] Performance band cut points computed and printed (section 3b)
- [ ] All 42 columns assigned a role (section 3c)
- [ ] Clustering input list printed for Person A
- [ ] Processed schema contract written (section 4)
- [ ] Team contract summary printed